# 6 · App-side float-threshold filtering · `hybrid_bitmap_taxonomy`

The bitmap gate handled "is this ad currently servable" cleanly. But ad
targeting also includes per-ad rules over the user's *float* interest
scores — expressions like `(electronics >= 0.80 OR luxury_retail >= 0.80)
AND fitness >= 0.50`. Those expressions can be different per campaign, and
the threshold values are continuous, not buckets — they don't fit into
SET membership or a global bitmap.

The mode for this is `hybrid_bitmap_taxonomy`: same retrieval as
`hybrid_bitmap_gating`, plus an app-side AND/OR/NOT evaluation against the
MAID's float scores after the bitmap gate.


In [1]:
from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## A campaign's `taxonomy_filter`

Every campaign carries an optional `taxonomy_filter` as a JSON AST. Leaves
are `{"gte": ["label", threshold]}`; internal nodes are `{"and": [...]}`,
`{"or": [...]}`, `{"not": <child>}`.


In [2]:
import json
raw = client.hgetall('campaign:c00042')
filter_ast = json.loads(raw['taxonomy_filter_json']) if raw.get('taxonomy_filter_json') else None
print(f'campaign:c00042 taxonomy_filter:')
print(json.dumps(filter_ast, indent=2))

campaign:c00042 taxonomy_filter:
{
  "and": [
    {
      "gte": [
        "tech",
        0.55
      ]
    },
    {
      "gte": [
        "foodie",
        0.35
      ]
    },
    {
      "not": {
        "or": [
          {
            "gte": [
              "luxury",
              0.35
            ]
          },
          {
            "gte": [
              "finance",
              0.45
            ]
          }
        ]
      }
    }
  ]
}


Rendered with operators inline, that filter reads like the kind of
expression in a real ad-platform's targeting UI.


In [3]:
def render(node):
    if node is None:
        return 'TRUE'
    if 'gte' in node:
        label, threshold = node['gte']
        return f'{label} >= {threshold}'
    if 'and' in node:
        return '(' + ' AND '.join(render(c) for c in node['and']) + ')'
    if 'or' in node:
        return '(' + ' OR '.join(render(c) for c in node['or']) + ')'
    if 'not' in node:
        return 'NOT ' + render(node['not'])
    raise ValueError(node)

print(render(filter_ast))

(tech >= 0.55 AND foodie >= 0.35 AND NOT (luxury >= 0.35 OR finance >= 0.45))


## The evaluator

`app.candidate.evaluate_taxonomy_filter` walks the AST against the user's
interest dict. It's about 15 lines of Python; nothing clever, nothing
slow.


In [4]:
import inspect
from app.candidate import evaluate_taxonomy_filter
print(inspect.getsource(evaluate_taxonomy_filter))

def evaluate_taxonomy_filter(node: object, scores: Mapping[str, float]) -> bool:
    """Evaluate a taxonomy filter AST against a user's float interest scores.

    Leaves are `{"gte": ["label", threshold]}` (label is missing → score 0).
    Internal nodes are `{"and": [...]}`, `{"or": [...]}`, `{"not": <child>}`.
    A `None` filter is treated as a match.
    """
    if node is None:
        return True
    if not isinstance(node, dict):
        raise TypeError(f"Unsupported taxonomy_filter node: {node!r}")
    if "gte" in node:
        label, threshold = node["gte"]
        return float(scores.get(label, 0.0)) >= float(threshold)
    if "and" in node:
        return all(evaluate_taxonomy_filter(child, scores) for child in node["and"])
    if "or" in node:
        return any(evaluate_taxonomy_filter(child, scores) for child in node["or"])
    if "not" in node:
        return not evaluate_taxonomy_filter(node["not"], scores)
    raise ValueError(f"Unsupported taxonomy_filter operator: {

## Where the gap shows up

Two MAIDs: one whose interest scores satisfy the filter, one whose don't.
This is the case `hybrid_bitmap_gating` would *miss* — the bitmap is built
from active+budget, it knows nothing about the user's score on `travel`.


In [5]:
from app.models import UserProfile

# Pull a couple of MAIDs and show how the same campaign's filter
# evaluates against each.
for maid_id in ['maid_00042', 'maid_00099', 'maid_00128']:
    user = UserProfile.from_redis_hash(client.hgetall(f'maid:{maid_id}'))
    interests_subset = {k: round(v, 2) for k, v in list(user.interests.items())[:6]}
    passes = evaluate_taxonomy_filter(filter_ast, user.interests)
    print(f'{maid_id}  passes campaign:c00042 filter? {passes}   '
          f'sample interests: {interests_subset}')

maid_00042  passes campaign:c00042 filter? False   sample interests: {'camping': 0.38, 'family': 0.21, 'finance': 0.68, 'fitness': 0.8, 'foodie': 0.39, 'gaming': 0.61}
maid_00099  passes campaign:c00042 filter? False   sample interests: {'camping': 0.63, 'family': 0.5, 'finance': 0.54, 'fitness': 0.23, 'foodie': 0.55, 'gaming': 0.35}
maid_00128  passes campaign:c00042 filter? True   sample interests: {'camping': 0.8, 'family': 0.31, 'finance': 0.44, 'fitness': 0.45, 'foodie': 0.4, 'gaming': 0.69}


## Running the bitmap+taxonomy mode end-to-end

`hybrid_bitmap_taxonomy` is `hybrid_bitmap_gating` plus the AST evaluation
on every surviving candidate. The extra cost is the evaluator running over
~30 candidates, in Python, in app memory.


In [6]:
from app.execution import execute_mode
from app.repository import RedisRepository
from app.models import HYBRID_BITMAP_MODE, HYBRID_BITMAP_TAXONOMY_MODE

repo = RedisRepository('redis://localhost:6381/0')

IDENTITY_TOKEN = 'id_00042_01'
maid_id, _ = repo.resolve_identity(IDENTITY_TOKEN)
scoring, _ = repo.fetch_scoring_profile(maid_id)

# Run both modes back-to-back so the comparison is apples-to-apples.
gating_only = execute_mode(
    repository=repo,
    maid_id=maid_id,
    mode=HYBRID_BITMAP_MODE,
    top_k=5,
    max_candidates=200,
    strong_signal_count=2,
    scoring_profile=scoring,
)
gating_plus_taxonomy = execute_mode(
    repository=repo,
    maid_id=maid_id,
    mode=HYBRID_BITMAP_TAXONOMY_MODE,
    top_k=5,
    max_candidates=200,
    strong_signal_count=2,
    scoring_profile=scoring,
)

import pandas as pd
pd.DataFrame([
    {
        'mode': gating_only.diagnostics.mode,
        'candidates': gating_only.diagnostics.final_candidate_count,
        'eligible': gating_only.diagnostics.eligible_count,
        'top_5_ids': [r.campaign_id for r in gating_only.top_results],
    },
    {
        'mode': gating_plus_taxonomy.diagnostics.mode,
        'candidates': gating_plus_taxonomy.diagnostics.final_candidate_count,
        'eligible': gating_plus_taxonomy.diagnostics.eligible_count,
        'top_5_ids': [r.campaign_id for r in gating_plus_taxonomy.top_results],
    },
])

                     mode  candidates  eligible                                 top_5_ids
0    hybrid_bitmap_gating          21        17  [c01011, c00848, c01551, c01222, c02229]
1  hybrid_bitmap_taxonomy          21        10  [c00848, c01551, c01222, c02229, c01617]

The candidate count is the same — both modes start from the same
bitmap-gated list. The eligible count drops because the taxonomy filter
removes campaigns whose AST is not satisfied by this MAID's interest
vector. Those are campaigns `hybrid_bitmap_gating` would have kept and
`hybrid_bitmap_taxonomy` correctly rejects.


## The headline comparison

Numbers below are the published offline ranking-quality comparison from
`reports/generated/evaluation.json`, using NDCG@K and the eligible-set
Jaccard against `full_realtime` as the truth set.


In [7]:
import json
from pathlib import Path
from notebooks._demo_setup import find_repo_root

repo_root = find_repo_root()
data = json.loads((repo_root / 'reports/generated/evaluation.json').read_text())
modes = data['synthetic_modes']['modes']

import pandas as pd
rows = []
for name, m in modes.items():
    rows.append({
        'mode': name,
        'NDCG@K': m['ndcg_at_k'],
        'top_jaccard_vs_full': m['top_result_jaccard_vs_full_realtime'],
        'eligible_set_jaccard_vs_full': m['eligible_set_jaccard_vs_full_realtime'],
        'avg_eligible': m['eligible_count'],
    })

frame = pd.DataFrame(rows)
print(frame.to_string(index=False))

                           mode  NDCG@K  top_jaccard_vs_full  eligible_set_jaccard_vs_full  avg_eligible
                  full_realtime  0.9818               1.0000                        1.0000       12.3803
            precomputed_segment  0.5584               0.3995                        0.5214       23.0940
hybrid_precompute_plus_realtime  0.9818               1.0000                        1.0000       12.3803
           hybrid_bitmap_gating  0.5584               0.3995                        0.5214       23.0940
         hybrid_bitmap_taxonomy  0.8751               0.8043                        0.8451       14.5214


And the published latency numbers from `hybrid_benchmark.json`
(serial benchmark on the tuned VM, N≈120, see methodology in
`reports/benchmark_report.md`):


In [8]:
bench = json.loads((repo_root / 'reports/generated/hybrid_benchmark.json').read_text())
loadtests = bench['loadtests']

rows = []
for name, m in loadtests.items():
    rows.append({
        'mode': name,
        'decision_p50_ms': m['decision_path_p50_latency_ms'],
        'decision_p99_ms': m['decision_path_p99_latency_ms'],
        'avg_round_trips': m['avg_mode_redis_round_trips'],
        'avg_sinter_ops': m['avg_sinter_ops'],
    })

frame = pd.DataFrame(rows).sort_values('decision_p50_ms')
print(frame.to_string(index=False))

                           mode  decision_p50_ms  decision_p99_ms  avg_round_trips  avg_sinter_ops
         hybrid_bitmap_taxonomy            3.723            9.916                3               0
           hybrid_bitmap_gating            3.991            5.332                3               0
            precomputed_segment            4.537           18.180                4               0
hybrid_precompute_plus_realtime            4.691           42.455                4               0
          maid_tightened_sinter            7.254           29.691                4               3
         maid_bruteforce_sinter           19.790           52.600               29              26
                  full_realtime          236.653          303.059                3               0


## What this tells the customer

- The precompute + bitmap pattern (notebook 4 + 5) gets the bid path to
  ~2 ms p50 on a tuned VM with no app-side targeting work beyond the
  frequency cap.
- The taxonomy filter (this notebook) adds the float-threshold AND/OR/NOT
  expression that the customer's bid model actually wants, with a small
  app-side cost.
- The end of the table — `hybrid_bitmap_taxonomy` — is the path that
  closes the gap between batch precompute and per-ad threshold-based
  targeting on continuous taxonomy scores.

Quality-wise: `hybrid_bitmap_taxonomy` lands an `NDCG@K` of `0.8751` vs
the `0.9818` reference. The remaining gap comes from the bitmap path
skipping the live `campaign_state:` fanout — same trade-off already
called out for `hybrid_bitmap_gating`. The customer can choose where to
sit on the latency-vs-quality curve.
